In [1]:
# -----------------------------
# Imports
# -----------------------------
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score,precision_score, recall_score, f1_score
from transformers import pipeline
from datasets import load_dataset
from openai import OpenAI

import os
from google.colab import userdata

from tqdm import tqdm
from torch.utils.data import DataLoader, WeightedRandomSampler, TensorDataset
import torch

In [2]:
# -----------------------------
# Load and preprocess dataset
# -----------------------------
ds = load_dataset("Mouwiya/drug-reviews")   # Hugging Face dataset
df = ds["train"].to_pandas().dropna(subset=["drugName", "condition", "review", "rating"])

# Collapse ratings into 5 buckets
def collapse_rating(r):
    if r in [1, 2]:
        return 0
    elif r in [3, 4]:
        return 1
    elif r in [5, 6]:
        return 2
    elif r in [7, 8]:
        return 3
    else:
        return 4

df["rating_bucket"] = df["rating"].apply(collapse_rating)

# Dictionary for reference
original_label_dict = {
    "0": "1-2 stars",
    "1": "3-4 stars",
    "2": "5-6 stars",
    "3": "7-8 stars",
    "4": "9-10 stars"
}

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.57M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/27703 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/46108 [00:00<?, ? examples/s]

# Text Cleaning Function

In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)         # remove HTML tags
    text = re.sub(r"['\"`]", "", text)         # remove quotes
    text = re.sub(r"[^a-z0-9.,!?;:() ]", " ", text)  # remove special characters
    text = re.sub(r"\s+", " ", text).strip()   # normalize whitespace
    return text

In [4]:
df["drugName"] = df["drugName"].apply(clean_text)
df["condition"] = df["condition"].apply(clean_text)
df["review"] = df["review"].apply(clean_text)


# EDA Section

Just printing useful EDA Analysis

In [5]:

print("EDA SUMMARY: ")
print("Dataset shape:", df.shape)
print("\nMissing values per column:\n", df.isna().sum())

print("\nRating bucket distribution:")
print(df["rating_bucket"].value_counts(normalize=True).round(3))

# Average review length
df["review_length"] = df["review"].apply(lambda x: len(x.split()))
print("\nAverage review length:", df["review_length"].mean())
print("Shortest review length:", df["review_length"].min())
print("Longest review length:", df["review_length"].max())

# Example cleaned text
print("\nSample cleaned review:\n", df["review"].iloc[0][:500], "...")

EDA SUMMARY: 
Dataset shape: (27703, 9)

Missing values per column:
 patient_id       0
drugName         0
condition        0
review           0
rating           0
date             0
usefulCount      0
review_length    0
rating_bucket    0
dtype: int64

Rating bucket distribution:
rating_bucket
4    0.479
3    0.183
0    0.170
2    0.094
1    0.074
Name: proportion, dtype: float64

Average review length: 95.24387250478287
Shortest review length: 28
Longest review length: 1181

Sample cleaned review:
 sober a year 8 25 11. god, aa and campral have worked. no cravings i couldnt handle. together all have helped me have a new lease on life. feel better, work has improved. highly recommend this medicine if you want to quit drinking. ...


# Feature Engineering
Creating a new column combining three features drugName + condition + review
Seperated with "|" character

In [ ]:
df["text"] = df["drugName"].astype(str) + " | " + df["condition"].astype(str) + " | " + df["review"].astype(str)
y = df["rating_bucket"].values

In [ ]:
# Stratified Split

train_df, val_df, y_train, y_val = train_test_split(
    df["text"], y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
df["text"].iloc[0]

### TF-IDF Vectorization

In [ ]:

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english")
X_train = tfidf.fit_transform(train_df)
X_val = tfidf.transform(val_df)

In [ ]:
# Weighted Random Sampler (for balancing)

class_counts = np.bincount(y_train)
class_weights = 1. / class_counts
sample_weights = class_weights[y_train]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

In [ ]:
# For completeness (if using in PyTorch pipelines later)
train_tensor = torch.tensor(X_train.toarray(), dtype=torch.float32)
val_tensor = torch.tensor(X_val.toarray(), dtype=torch.float32)
train_labels_tensor = torch.tensor(y_train, dtype=torch.long)
val_labels_tensor = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(train_tensor, train_labels_tensor)
val_dataset = TensorDataset(val_tensor, val_labels_tensor)

train_loader = DataLoader(train_dataset, sampler=sampler, batch_size=64)

print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

In [ ]:
# -----------------------------
# Choose Model
# -----------------------------
# Logistic Regression
#model = LogisticRegression(max_iter=300, class_weight="balanced")
#model.fit(X_train, y_train)
#y_pred = model.predict(X_val)
#print("Accuracy:", accuracy_score(y_val, y_pred))
#print("\nClassification Report:\n", classification_report(y_val, y_pred, target_names=[original_label_dict[str(i)] for i in range(5)]))
#Accuracy: 0.5356433856704566

# Linear SVM
#model = LinearSVC(class_weight="balanced")

#model.fit(X_train, y_train)
#y_pred = model.predict(X_val)
#print("Accuracy:", accuracy_score(y_val, y_pred))
#print("\nClassification Report:\n", classification_report(y_val, y_pred, target_names=[original_label_dict[str(i)] for i in range(5)]))
#Accuracy: 0.5663237682728749

In [ ]:
# Option 2: Random Forest
model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
# -----------------------------
# Train & Evaluate
# -----------------------------
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
print("Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred, target_names=[original_label_dict[str(i)] for i in range(5)]))

# Zero-Shot model

In [86]:
df[:10]

,patient_id,drugName,condition,review,rating,date,usefulCount,review_length,rating_bucket,text
0,191114,campral,alcohol dependence,"sober a year 8 25 11. god, aa and campral have...",10.0,"September 3, 2011",33,43,4,campral | alcohol dependence | sober a year 8 ...
1,142693,levonorgestrel,birth control,ive been on birth control for a while now due ...,4.0,"August 9, 2017",3,143,1,levonorgestrel | birth control | ive been on b...
2,71561,vraylar,bipolar disorde,"hi, this is an updated experience. i have now ...",8.0,"August 16, 2016",12,131,3,"vraylar | bipolar disorde | hi, this is an upd..."
3,25765,ethinyl estradiol norelgestromin,birth control,i have been on the ortho evra patch for just o...,8.0,"September 15, 2013",16,140,3,ethinyl estradiol norelgestromin | birth contr...
4,12843,etanercept,psoriasis,i have been on enbrel for 7 years and i have t...,9.0,"August 5, 2010",9,66,4,etanercept | psoriasis | i have been on enbrel...
5,60200,nuvaring,birth control,i have just started nuvaring a week ago today....,6.0,"April 19, 2012",0,87,2,nuvaring | birth control | i have just started...
6,101305,aubra,birth control,the only good thing about this medication is t...,1.0,"March 21, 2017",1,133,0,aubra | birth control | the only good thing ab...
7,20928,acyclovir,cold sores,ive suffered with cold sores since i was about...,10.0,"July 11, 2016",7,118,4,acyclovir | cold sores | ive suffered with col...
8,112088,gabapentin,hot flashes,"i was prescribed gabapentin, 300mg three times...",1.0,"July 15, 2017",6,67,0,gabapentin | hot flashes | i was prescribed ga...
9,84867,ethinyl estradiol norgestimate,birth control,the triphasal birth control pills have been a ...,10.0,"March 21, 2016",0,84,4,ethinyl estradiol norgestimate | birth control...


In [87]:
# Initialize zero-shot classifier
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)
candidate_labels = list(original_label_dict.values())
true_labels, predicted_labels = [], []
sampleCnt = 10
df_zeroshot = df[:sampleCnt]

for _, row in tqdm(df_zeroshot.iterrows(), total=sampleCnt):
    text = row["text"]
    result = classifier(text, candidate_labels)
    pred_label = candidate_labels.index(result["labels"][0])
    predicted_labels.append(pred_label)
    true_labels.append(row["rating_bucket"])


# Evaluate
print("=== Zero-Shot Classification Results ===")
print(true_labels, predicted_labels)
print("Accuracy:", accuracy_score(true_labels, predicted_labels))
print("\nClassification Report:\n", classification_report(true_labels, predicted_labels))

Device set to use cpu
100%|██████████| 10/10 [02:05<00:00, 12.55s/it]

=== Zero-Shot Classification Results ===
[4, 1, 3, 3, 4, 2, 0, 4, 0, 4] [3, 0, 0, 0, 3, 3, 0, 0, 0, 0]
Accuracy: 0.2

Classification Report:
               precision    recall  f1-score   support

           0       0.29      1.00      0.44         2
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         1
           3       0.00      0.00      0.00         2
           4       0.00      0.00      0.00         4

    accuracy                           0.20        10
   macro avg       0.06      0.20      0.09        10
weighted avg       0.06      0.20      0.09        10




/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
result["labels"][0]

In [ ]:
# Initialize zero-shot classifier
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)

true_labels, predicted_labels = [], []

for _, row in tqdm(df.iterrows(), total=len(df)):
    text = row["text"]
    result = classifier(text, labels)
    pred_label = labels.index(result["labels"][0])
    predicted_labels.append(pred_label)
    true_labels.append(row["rating_bucket"])
    break

# Evaluate
print("=== Zero-Shot Classification Results ===")
print(true_labels, predicted_labels)
print("Accuracy:", accuracy_score(true_labels, predicted_labels))
print("\nClassification Report:\n", classification_report(true_labels, predicted_labels))


# Transformer Model Experiement

Insert your code here